# H2 Phase 1 — Synthetic Data Generation

**Accelerator:** None (CPU)

Bu notebook 300K synthetic el yazısı kelime görseli üretir.

**Tahmini süre:** ~3 saat (CPU)

**Çıktı:** `synthetic_data.zip` → Phase 2 notebook'una dataset olarak ekleyeceksin.

---
**Donanım (makale için):**
- CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
- RAM: 13 GB

In [ ]:
# Hücre 1: crnn-h2-code dataset'ten scripts'leri kopyala
import sys, os, shutil

CODE_INPUT = "/kaggle/input/datasets/brht25/crnn-h2-code"

if os.path.exists(CODE_INPUT):
    os.makedirs("/kaggle/working/cloud", exist_ok=True)
    for fname in os.listdir(f"{CODE_INPUT}/cloud"):
        if fname.endswith((".py", ".txt", ".sh")):
            shutil.copy(f"{CODE_INPUT}/cloud/{fname}", f"/kaggle/working/cloud/{fname}")
    shutil.copy(f"{CODE_INPUT}/trigram_lm.py", "/kaggle/working/trigram_lm.py")
    print("Scripts kopyalandı OK")
else:
    print(f"⚠️  {CODE_INPUT} bulunamadı!")
    print("Çözüm: Notebook'a ekle → Add Data → Your Datasets → crnn-h2-code")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Hücre 2: Bağımlılıkları kur
# Not: Pillow 9.5.0 gerekiyor — trdg 1.8.0 kaldırılan getsize() API'sini kullanıyor (Pillow 10+)
!pip install -q arabic-reshaper==2.1.4 python-bidi==0.4.2 diffimg==0.2.3 wikipedia nltk==3.8.1 "Pillow==9.5.0" opencv-python-headless
!pip install -q trdg==1.8.0 --no-deps

In [ ]:
# Hücre 3: Google Fonts indir (15 el yazısı font)
import os, urllib.request

FONT_DIR = "/kaggle/working/fonts"
os.makedirs(FONT_DIR, exist_ok=True)

FONTS = {
    "Caveat.ttf":            "https://github.com/google/fonts/raw/main/ofl/caveat/Caveat-Regular.ttf",
    "IndieFlower.ttf":       "https://github.com/google/fonts/raw/main/ofl/indieflower/IndieFlower-Regular.ttf",
    "Kalam.ttf":             "https://github.com/google/fonts/raw/main/ofl/kalam/Kalam-Regular.ttf",
    "PatrickHand.ttf":       "https://github.com/google/fonts/raw/main/ofl/patrickhand/PatrickHand-Regular.ttf",
    "ShadowsIntoLight.ttf":  "https://github.com/google/fonts/raw/main/ofl/shadowsintolight/ShadowsIntoLight.ttf",
    "ArchitectsDaughter.ttf":"https://github.com/google/fonts/raw/main/ofl/architectsdaughter/ArchitectsDaughter.ttf",
    "DancingScript.ttf":     "https://github.com/google/fonts/raw/main/ofl/dancingscript/DancingScript-Regular.ttf",
    "Satisfy.ttf":           "https://github.com/google/fonts/raw/main/ofl/satisfy/Satisfy-Regular.ttf",
    "GloriaHallelujah.ttf":  "https://github.com/google/fonts/raw/main/ofl/gloriahallelujah/GloriaHallelujah.ttf",
    "Handlee.ttf":           "https://github.com/google/fonts/raw/main/ofl/handlee/Handlee-Regular.ttf",
    "PermanentMarker.ttf":   "https://github.com/google/fonts/raw/main/ofl/permanentmarker/PermanentMarker-Regular.ttf",
    "Sacramento.ttf":        "https://github.com/google/fonts/raw/main/ofl/sacramento/Sacramento-Regular.ttf",
    "Itim.ttf":              "https://github.com/google/fonts/raw/main/ofl/itim/Itim-Regular.ttf",
    "JustAnotherHand.ttf":   "https://github.com/google/fonts/raw/main/ofl/justanotherhand/JustAnotherHand-Regular.ttf",
    "Pacifico.ttf":          "https://github.com/google/fonts/raw/main/ofl/pacifico/Pacifico-Regular.ttf",
}

ok = 0
for name, url in FONTS.items():
    dest = os.path.join(FONT_DIR, name)
    if os.path.exists(dest):
        ok += 1
        continue
    try:
        urllib.request.urlretrieve(url, dest)
        ok += 1
    except Exception as e:
        print(f"  WARN: {name} indirilemedi — {e}")

print(f"{ok}/{len(FONTS)} font hazır → {FONT_DIR}")

In [ ]:
# Hücre 4: Synthetic data üret
!python cloud/phase1_synthetic_gen.py \
    --count 300000 \
    --output /kaggle/working/synthetic_data \
    --font-dir /kaggle/working/fonts \
    --batch-size 10000

In [ ]:
# Hücre 5: Doğrulama + zip (Phase 2'ye transfer için)
import glob

imgs = glob.glob("/kaggle/working/synthetic_data/words/*.png")
print(f"Toplam PNG: {len(imgs):,}")

with open("/kaggle/working/synthetic_data/labels.txt") as f:
    n_labels = sum(1 for _ in f)
print(f"Labels satır: {n_labels:,}")

# Zip oluştur — Kaggle output olarak indir, sonra Phase 2'de dataset ekle
print("\nZip oluşturuluyor (bu adım ~5-10 dk sürer)...")
!cd /kaggle/working && zip -q -r synthetic_data.zip synthetic_data/
print("synthetic_data.zip hazır — Output sekmesinden indir veya dataset olarak kaydet")

In [ ]:
# Hücre 6: Preview grid (30 random sample)
import random, cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

samples = []
with open("/kaggle/working/synthetic_data/labels.txt") as f:
    lines = f.readlines()
random.shuffle(lines)
for line in lines[:30]:
    parts = line.strip().split(" ", 1)
    if len(parts) == 2:
        fname, word = parts
        img = cv2.imread(f"/kaggle/working/synthetic_data/words/{fname}", cv2.IMREAD_GRAYSCALE)
        if img is not None:
            samples.append((img, word))

fig, axes = plt.subplots(5, 6, figsize=(18, 8))
for i, ax in enumerate(axes.flat):
    if i < len(samples):
        ax.imshow(samples[i][0], cmap="gray")
        ax.set_title(samples[i][1], fontsize=8)
    ax.axis("off")
plt.suptitle("Synthetic Data Preview (30 samples)", fontsize=12)
plt.tight_layout()
plt.savefig("/kaggle/working/preview_grid.png", dpi=100)
plt.show()
print("Preview kaydedildi: /kaggle/working/preview_grid.png")